# Fine-tuning YOLOv8n on VisDrone

YOLOv8n ships trained on **COCO**: ordinary photos, taken at eye level, of 80 kinds of object.
**VisDrone** is the opposite: 10 classes of people and vehicles seen **from a drone**, looking down,
and usually tiny. This notebook measures how badly the COCO model does on that footage, fine-tunes
it on VisDrone, and measures it again the same way.

What it does:

1. **Setup** - check for a GPU, install a pinned Ultralytics, mount Google Drive.
2. **Data** - use Ultralytics' own `VisDrone.yaml`, which downloads the dataset and converts it.
3. **Baseline** - score the COCO model *before* training, only on the classes COCO actually has.
4. **Train** - 50 epochs at 640 px with a fixed seed. Training can pick up again after a disconnect.
5. **Evaluate** - mAP50, mAP50-95 and AP per class, on the shared subset and on all 10 classes.
6. **Save** - write a JSON record in the same shape as the repo's `results/`, and export `best.pt` plus an ONNX copy.

**How to run:** Runtime -> Change runtime type -> **GPU** (a T4 is enough), then Runtime -> **Run all**.

**Colab will disconnect.** Everything that matters is written to Drive under
`MyDrive/jarvis/visdrone_yolov8n/`. After a disconnect, reconnect and **Run all** again. That is always
safe: the dataset downloads again (it lives on the VM's fast local disk, and a copy is not worth
protecting), the baseline runs again (about a minute), and training **resumes** from the last saved
epoch rather than starting over. If training has already finished, the cell sees that and skips it.

## Step 1 - Setup

### 1a. Is there a GPU?

Training a detector on a CPU would take days. Rather than let that happen silently, this cell
**stops the notebook with an error** when no GPU is attached. Nothing further down runs, so you
never end up with a half-finished CPU run that you then have to throw away.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU found - refusing to continue. This notebook trains a detector and will not fall back to CPU.\n"
        "Fix: Runtime > Change runtime type > Hardware accelerator: GPU (T4 is enough), then Runtime > Run all."
    )

GPU_NAME = torch.cuda.get_device_name(0)
print(f"GPU: {GPU_NAME} | torch {torch.__version__} | CUDA {torch.version.cuda}")
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

### 1b. Install Ultralytics (pinned)

The version is **pinned**, so running this notebook again months from now trains with the same code.
The default training settings, the data augmentation and even the mAP arithmetic all change between
Ultralytics releases, and any of those changes would make two runs impossible to compare. The
installed versions are also written into the results JSON.

`onnx`, `onnxslim` and `onnxruntime` are needed for the export and its check in Step 6.

In [ ]:
%pip install -q "ultralytics==8.4.60" onnx onnxslim onnxruntime

### 1c. Mount Drive and decide where things live

There are two kinds of storage here, and the difference matters:

| What | Where | Why |
|---|---|---|
| Checkpoints, logs, plots, results JSON, exports | **Google Drive** | These survive a disconnect. Losing them means losing hours of training. |
| The VisDrone images (~2.3 GB) | **`/content`**, the VM's local disk | Much faster to read than Drive, and can be downloaded again in a few minutes. |

Drive is mounted through a network filesystem. A training loop that read about 6,500 images from it
every epoch would spend most of its time waiting on I/O. The one file that must survive,
`last.pt`, is small (about 6 MB) and is written once per epoch, so saving it to Drive costs almost nothing.

`sessions.jsonl` records the GPU used by each Colab session. A resumed run can finish on a different
GPU type from the one it started on, and the results record should show that rather than hide it.

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/jarvis/visdrone_yolov8n")
RUNS_DIR = DRIVE_ROOT / "runs"  # Ultralytics training output (weights/last.pt, best.pt, results.csv, plots)
EVAL_DIR = DRIVE_ROOT / "eval"  # validation plots
EXPORT_DIR = DRIVE_ROOT / "export"  # best.pt + ONNX copy, for the speed step
RESULTS_JSON = DRIVE_ROOT / "results" / "benchmarks_visdrone.json"
SESSIONS_LOG = DRIVE_ROOT / "sessions.jsonl"
for d in (RUNS_DIR, EVAL_DIR, EXPORT_DIR, RESULTS_JSON.parent):
    d.mkdir(parents=True, exist_ok=True)

LOCAL_DATASETS = Path("/content/datasets")  # local disk, deliberately NOT Drive

# ---- Experiment settings -------------------------------------------------
SEED = 42
EPOCHS = 50
IMGSZ = 640  # the speed target's input size - see the note in Step 4 before changing it
BATCH = 16  # fixed, not AutoBatch (-1): AutoBatch picks per-GPU, which would make runs differ
RUN_NAME = f"yolov8n_visdrone_{IMGSZ}_e{EPOCHS}_s{SEED}"

# ---- Evaluation settings (Ultralytics' own val defaults, so numbers are comparable) ----
CONF = 0.001  # keep almost every box: AP sweeps the threshold itself
NMS_IOU = 0.7
MAX_DET = 300

with SESSIONS_LOG.open("a") as f:
    f.write(json.dumps({"timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
                        "gpu": GPU_NAME}) + "\n")
print("Outputs ->", DRIVE_ROOT)

## Step 2 - Data

Ultralytics has a dataset file for VisDrone built in. The first time it is used, it downloads the
original VisDrone archives and **converts** their annotations to YOLO format, one `.txt` per image
with a line `class x_center y_center width height` per object (all normalised to 0-1). The raw
VisDrone format also has "ignored regions" (crowded areas the annotators did not label box by box);
the converter drops those.

The only change made to the built-in file is its `path`. It is pinned to `/content/datasets/VisDrone`
so the images land on local disk. Everything else, including the class list and the download script,
is Ultralytics' own. The cell then **checks that the class order is the one this notebook assumes**,
because all of the class mapping in Step 3 is done by index, and a silent reordering would produce
wrong numbers with no error.

The **val** split (548 images) is used for every number reported here. VisDrone's test-dev split has
labels too, but val is what Ultralytics validates on during training, so val keeps every number in
this notebook on the same images.

In [ ]:
import numpy as np
import pandas as pd
import ultralytics
import yaml
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset, img2label_paths
from ultralytics.utils.checks import check_yaml

VISDRONE_NAMES = ["pedestrian", "people", "bicycle", "car", "van",
                  "truck", "tricycle", "awning-tricycle", "bus", "motor"]

builtin_yaml = Path(check_yaml("VisDrone.yaml"))
cfg = yaml.safe_load(builtin_yaml.read_text())
cfg["path"] = str(LOCAL_DATASETS / "VisDrone")  # the only override
DATA_YAML = Path("/content/VisDrone.yaml")
DATA_YAML.write_text(yaml.safe_dump(cfg, sort_keys=False))

data = check_det_dataset(str(DATA_YAML))  # first call downloads + converts (~2.3 GB, a few minutes)

names = [data["names"][i] for i in range(len(data["names"]))]
assert names == VISDRONE_NAMES, f"VisDrone class order changed upstream: {names}. Fix the mappings in Step 3."

val = data["val"]
VAL_DIR = Path(val[0] if isinstance(val, list) else val)
VAL_IMAGES = sorted(str(p) for p in VAL_DIR.glob("*.jpg"))
print(f"Ultralytics {ultralytics.__version__} | val images: {len(VAL_IMAGES)} in {VAL_DIR}")
if len(VAL_IMAGES) != 548:
    print(f"WARNING: expected 548 VisDrone val images, found {len(VAL_IMAGES)}")

### Load the ground truth ourselves

The baseline in Step 3 needs an evaluator that can merge and ignore classes, which Ultralytics'
`val()` cannot do, so the label files are read directly here.

A defensive check: any label with a class id outside 0-9 (the raw VisDrone data has an 11th "others"
category) makes Ultralytics treat the *whole image* as corrupt and skip it in validation. To keep the
two evaluators on the same images, this loader skips those images as well, and records which ones it
skipped. Normally the list is empty.

In [ ]:
def load_ground_truth(image_paths, num_classes):
    # Returns ({image_stem: {"cls": int[N], "xywhn": float[N, 4]}}, {skipped_stem: [bad class ids]})
    gts, skipped = {}, {}
    for img, lbl in zip(image_paths, img2label_paths(image_paths)):
        lbl = Path(lbl)
        rows = np.loadtxt(lbl, ndmin=2) if lbl.exists() and lbl.stat().st_size else np.zeros((0, 5))
        cls = rows[:, 0].astype(int)
        bad = (cls < 0) | (cls >= num_classes)
        if bad.any():
            skipped[Path(img).stem] = sorted(set(cls[bad].tolist()))
            continue
        gts[Path(img).stem] = {"cls": cls, "xywhn": rows[:, 1:5]}
    return gts, skipped


GT, SKIPPED = load_ground_truth(VAL_IMAGES, len(VISDRONE_NAMES))
gt_counts = np.bincount(np.concatenate([g["cls"] for g in GT.values()]), minlength=len(VISDRONE_NAMES))
print(f"{len(GT)} images, {gt_counts.sum()} boxes | skipped images: {SKIPPED or 'none'}")
pd.DataFrame({"class": VISDRONE_NAMES, "val instances": gt_counts})

## Step 3 - Baseline: the COCO model, before any training

Without a "before" number, the fine-tuned result has nothing to be compared against. The difficulty
is that **the two models do not share a label set.** The COCO model predicts COCO's 80 classes and
knows nothing about VisDrone's 10. Only the classes that *mean the same thing* can be scored.

| VisDrone class | COCO class | Scored in the baseline? |
|---|---|---|
| pedestrian | person | yes - merged into **person** (see below) |
| people | person | yes - merged into **person** |
| bicycle | bicycle | yes |
| car | car | yes |
| truck | truck | yes |
| bus | bus | yes |
| motor | motorcycle | yes |
| **van** | *none* | **no - COCO has no van class** |
| **tricycle** | *none* | **no - COCO has no tricycle class** |
| **awning-tricycle** | *none* | **no - COCO has no awning-tricycle class** |

**These three classes are deliberately left out of the baseline.** They are not scored as zero,
because a zero would read as "the model tried and failed" when in fact it never had those classes to
predict. They only appear in the fine-tuned model's all-10-class numbers in Step 5.

### Why pedestrian and people are merged

VisDrone divides humans **by pose**: `pedestrian` is someone standing or walking, and `people` is
anyone else (sitting, lying, riding a bike). COCO has a single `person` class. If every COCO person
detection were copied into both classes, a correct detection of a seated person would count as a
*false positive* under `pedestrian`. That would penalise the COCO model for a distinction it was never
trained to make. So both are merged into one **person** class, in the ground truth *and* in the
fine-tuned model's predictions when it is scored on this subset. The subset therefore has **6 scored
classes**: person, bicycle, car, truck, bus, motor.

Merging creates one new problem. The fine-tuned model can box the same human as both `pedestrian`
and `people`, and after the merge those are two `person` boxes, one of which would count as a false
positive. So NMS runs again after the remap, with the same IoU threshold of 0.7 as everywhere else.

### Why the unscored classes are treated as "ignore" regions

COCO has no van class, so the COCO model will label a van as `car` or `truck`. The van is not a car in
the ground truth, so each of those boxes would be scored as a *false positive* against the car class.
The baseline would lose points on vans, a class this notebook has just said it will not score.

The fix follows COCO's own convention for regions nobody should be scored on: **a prediction that does
not match any scored box, but overlaps a van, tricycle or awning-tricycle box, is ignored.** It counts
as neither a hit nor a miss. The **same rule applies to both models.** To show how much the rule
changes the outcome, the numbers without it are also reported.

### How AP is computed (the evaluator used on the subset)

* **IoU** (intersection over union) measures how much a predicted box overlaps a ground-truth box, from
  0 (no overlap) to 1 (identical).
* For each class and each IoU threshold, predictions are taken **in order of confidence**. Each one
  claims the unclaimed ground-truth box it overlaps most, if that overlap is at or above the
  threshold. A prediction that claims a box is a true positive; one that finds nothing is a false
  positive.
* Moving down that ordered list traces out a **precision-recall curve**. **AP** is the area under it,
  using COCO's 101-point interpolation.
* **mAP50** is AP at IoU 0.5, averaged over classes: it asks whether the box is roughly in the right
  place. **mAP50-95** averages AP over IoU 0.50, 0.55, ..., 0.95: it also rewards boxes that fit
  *tightly*, which is much harder when an object is 12 pixels wide.

This evaluator is a small reimplementation, so Step 5 **checks it against Ultralytics' own `val()`**
on all 10 classes. If the two agree there, its subset numbers can be trusted.

In [ ]:
import torchvision

IOU_THRESHOLDS = np.linspace(0.5, 0.95, 10)

# ---- The mapped subset -----------------------------------------------------------------
SUBSET_NAMES = ["person", "bicycle", "car", "truck", "bus", "motor"]
COCO_TO_SUBSET = {0: 0, 1: 1, 2: 2, 7: 3, 5: 4, 3: 5}  # person, bicycle, car, truck, bus, motorcycle
VISDRONE_TO_SUBSET = {0: 0, 1: 0, 2: 1, 3: 2, 5: 3, 8: 4, 9: 5}  # pedestrian+people -> person
VISDRONE_UNSCORED = {4: "van", 6: "tricycle", 7: "awning-tricycle"}  # no COCO equivalent -> ignore regions
IDENTITY_10 = {i: i for i in range(len(VISDRONE_NAMES))}


def predict_all(weights, image_paths, classes=None):
    # Run a model over every image; keep raw boxes in the model's OWN class ids.
    model = YOLO(weights)
    out = {}
    for r in model.predict(source=image_paths, stream=True, imgsz=IMGSZ, conf=CONF, iou=NMS_IOU,
                           max_det=MAX_DET, classes=classes, device=0, batch=16, verbose=False):
        b = r.boxes
        out[Path(r.path).stem] = {"xyxy": b.xyxy.cpu().numpy(), "conf": b.conf.cpu().numpy(),
                                  "cls": b.cls.cpu().numpy().astype(int), "shape": r.orig_shape}
    return out


def to_eval_space(preds, class_map):
    # Relabel predictions into evaluation class ids, drop unmapped ones, and re-run NMS so that
    # two source classes merged into one (pedestrian + people -> person) can't double-count.
    out = {}
    for stem, p in preds.items():
        new_cls = np.array([class_map.get(int(c), -1) for c in p["cls"]], dtype=int)
        keep = new_cls >= 0
        xyxy, conf, cls = p["xyxy"][keep], p["conf"][keep], new_cls[keep]
        if len(xyxy):
            idx = torchvision.ops.batched_nms(torch.from_numpy(xyxy).float(), torch.from_numpy(conf).float(),
                                              torch.from_numpy(cls), NMS_IOU).numpy()[:MAX_DET]
            xyxy, conf, cls = xyxy[idx], conf[idx], cls[idx]
        out[stem] = {"xyxy": xyxy, "conf": conf, "cls": cls, "shape": p["shape"]}
    return out


def xywhn_to_xyxy(xywhn, h, w):
    xc, yc, bw, bh = xywhn[:, 0] * w, xywhn[:, 1] * h, xywhn[:, 2] * w, xywhn[:, 3] * h
    return np.stack([xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2], axis=1)


def box_iou(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    tl = np.maximum(a[:, None, :2], b[None, :, :2])
    br = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.clip(br - tl, 0, None).prod(2)
    area_a = (a[:, 2:] - a[:, :2]).prod(1)
    area_b = (b[:, 2:] - b[:, :2]).prod(1)
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)


def match(pred_boxes, gt_boxes, ignore_boxes):
    # Greedy COCO matching for ONE image and ONE class. pred_boxes must be sorted by confidence (desc).
    # Returns tp[P, T] and ignored[P, T] booleans, one column per IoU threshold.
    P, T = len(pred_boxes), len(IOU_THRESHOLDS)
    tp, ignored = np.zeros((P, T), bool), np.zeros((P, T), bool)
    if P == 0:
        return tp, ignored
    iou = box_iou(pred_boxes, gt_boxes)
    ign_iou = box_iou(pred_boxes, ignore_boxes).max(1) if len(ignore_boxes) else np.zeros(P)
    for t, thr in enumerate(IOU_THRESHOLDS):
        taken = np.zeros(len(gt_boxes), bool)
        for i in range(P):
            if len(gt_boxes):
                cand = np.where(~taken & (iou[i] >= thr), iou[i], -1.0)
                j = int(cand.argmax())
                if cand[j] >= thr:
                    tp[i, t], taken[j] = True, True
                    continue
            if ign_iou[i] >= thr:
                ignored[i, t] = True  # sits on an unscored object: neither hit nor miss
    return tp, ignored


def average_precision(conf, tp, n_gt):
    # COCO 101-point interpolated AP for one class at one IoU threshold.
    if n_gt == 0:
        return float("nan")
    order = np.argsort(-conf, kind="stable")
    tp = tp[order]
    ctp, cfp = np.cumsum(tp), np.cumsum(~tp)
    recall = ctp / n_gt
    precision = ctp / np.maximum(ctp + cfp, 1e-9)
    precision = np.maximum.accumulate(precision[::-1])[::-1]  # envelope: best precision at >= this recall
    idx = np.searchsorted(recall, np.linspace(0, 1, 101), side="left")
    return float(np.mean([precision[i] if i < len(precision) else 0.0 for i in idx]))


def evaluate(preds, gts, gt_map, ignore_gt_classes, class_names):
    # preds are already in eval space (see to_eval_space). Returns ap[C, T] and n_gt[C].
    C, T = len(class_names), len(IOU_THRESHOLDS)
    acc = {c: {"conf": [], "tp": [], "ign": []} for c in range(C)}
    n_gt = np.zeros(C, int)
    ignore_ids = list(ignore_gt_classes)
    for stem, g in gts.items():
        p = preds[stem]
        h, w = p["shape"]
        g_xyxy = xywhn_to_xyxy(g["xywhn"], h, w)
        g_cls = np.array([gt_map.get(int(c), -1) for c in g["cls"]], dtype=int)
        ign_boxes = g_xyxy[np.isin(g["cls"], ignore_ids)]
        for c in range(C):
            gb = g_xyxy[g_cls == c]
            n_gt[c] += len(gb)
            m = p["cls"] == c
            order = np.argsort(-p["conf"][m], kind="stable")
            tp, ign = match(p["xyxy"][m][order], gb, ign_boxes)
            acc[c]["conf"].append(p["conf"][m][order])
            acc[c]["tp"].append(tp)
            acc[c]["ign"].append(ign)
    ap = np.full((C, T), np.nan)
    for c in range(C):
        conf = np.concatenate(acc[c]["conf"])
        tp = np.concatenate(acc[c]["tp"]).reshape(-1, T)
        ign = np.concatenate(acc[c]["ign"]).reshape(-1, T)
        for t in range(T):
            keep = ~ign[:, t]
            ap[c, t] = average_precision(conf[keep], tp[keep, t], n_gt[c])
    return ap, n_gt


def summarize(ap, n_gt, class_names):
    return {
        "map50": float(np.nanmean(ap[:, 0])),
        "map50_95": float(np.nanmean(ap.mean(1))),
        "per_class": [{"class": n, "gt_instances": int(k), "ap50": float(ap[i, 0]), "ap50_95": float(ap[i].mean())}
                      for i, (n, k) in enumerate(zip(class_names, n_gt))],
    }

### Run the baseline

`classes=[...]` makes the COCO model return **only** the six mapped COCO classes. Without it, boxes
for classes that are about to be discarded (traffic lights, boats and so on) would use up part of the
300-box `max_det` budget in crowded images, and the baseline would lose real detections for no reason.

In [ ]:
baseline_raw = predict_all("yolov8n.pt", VAL_IMAGES, classes=sorted(COCO_TO_SUBSET))
baseline_eval = to_eval_space(baseline_raw, COCO_TO_SUBSET)

ap, n = evaluate(baseline_eval, GT, VISDRONE_TO_SUBSET, VISDRONE_UNSCORED, SUBSET_NAMES)
BASELINE_SUBSET = summarize(ap, n, SUBSET_NAMES)
ap, n = evaluate(baseline_eval, GT, VISDRONE_TO_SUBSET, (), SUBSET_NAMES)  # sensitivity: no ignore rule
BASELINE_SUBSET_NO_IGNORE = summarize(ap, n, SUBSET_NAMES)

print(f"Baseline (COCO yolov8n), mapped subset: mAP50 {BASELINE_SUBSET['map50']:.4f} | "
      f"mAP50-95 {BASELINE_SUBSET['map50_95']:.4f}")
print(f"  without the ignore rule:             mAP50 {BASELINE_SUBSET_NO_IGNORE['map50']:.4f} | "
      f"mAP50-95 {BASELINE_SUBSET_NO_IGNORE['map50_95']:.4f}")
print("Not scored (no COCO equivalent):", ", ".join(VISDRONE_UNSCORED.values()))
pd.DataFrame(BASELINE_SUBSET["per_class"]).round(4)

Expect these numbers to be **low**. That is the point of measuring them. A COCO person is usually
photographed from the side and fills a good part of the frame. A VisDrone pedestrian is seen from
above, from tens of metres up, and is often only 10-20 pixels tall. The object is the same but it
looks nothing alike, and that gap is what fine-tuning has to close.

## Step 4 - Train

**Fine-tuning** means training does not start from random weights. It starts from `yolov8n.pt`,
which already knows edges, textures and shapes from COCO, and continues on VisDrone. Ultralytics
notices that the dataset has 10 classes rather than 80 and **re-initialises only the class-prediction
layers** of the detection head. The log line `Transferred X/Y items from pretrained weights` shows
this: nearly everything is carried over, and the few layers that are not are the ones sized for 80
classes.

Settings, and why each one:

* **`epochs=50`**: one epoch is one full pass over the 6,471 training images.
* **`seed=42`, `deterministic=True`**: fixes the random choices (data shuffling, augmentation) so a
  rerun follows the same path. *Caveat:* some GPU kernels are not bit-exact even with this setting,
  and a different GPU model will diverge slightly. The results are reproducible to within a small
  amount of noise, not to every digit.
* **`batch=16`**: fixed. AutoBatch (`-1`) chooses a batch size from the GPU's memory, so a T4 and an
  L4 would train different experiments.
* **`patience=EPOCHS`**: turns early stopping off in effect, so the run is exactly 50 epochs. The
  record then says what happened rather than "up to 50".

### Why imgsz stays at 640

> **Larger input sizes help small objects, and VisDrone is mostly small objects**, so training at 960
> or 1280 would very likely score noticeably higher. But the input size is also **what the detector
> costs to run**: compute grows roughly with the square of the side, so 1280 is about 4x the work of
> 640, and fps falls to match. 640 is the input size of the speed target that the next step
> benchmarks, and a model trained at 640 is the one that will actually run at that speed. Training
> larger and then running at 640 does not solve this: the model learns object sizes at the scale it
> was trained on. If small-object accuracy turns out to be the limit, make it its own experiment,
> with its fps measured next to its mAP.

### Resuming

Every epoch, Ultralytics writes `weights/last.pt` (the model plus the optimizer state, the
learning-rate schedule position and the epoch number) to the run directory on Drive. The cell below
checks that file:

* **no `last.pt`**: start fresh from `yolov8n.pt`;
* **`last.pt` part-way through**: `resume=True` continues from the next epoch with the original
  arguments, which are read back out of the checkpoint;
* **finished**: skip. When Ultralytics completes the final epoch it strips the checkpoint and marks
  it `epoch = -1`, and that marker is how this cell knows. Calling `resume=True` on a finished run
  raises an error, which is why the cell checks first.

On a T4, expect this to take roughly a few hours. Colab sessions can end before that, and that is
what the resume path is for.

In [ ]:
RUN_DIR = RUNS_DIR / RUN_NAME
LAST_PT = RUN_DIR / "weights" / "last.pt"
BEST_PT = RUN_DIR / "weights" / "best.pt"


def training_state():
    if not LAST_PT.exists():
        return "fresh", 0
    ckpt = torch.load(LAST_PT, map_location="cpu", weights_only=False)
    epoch = ckpt.get("epoch", -1)  # 0-based index of the last COMPLETED epoch; -1 once finalised
    if epoch == -1 or epoch + 1 >= EPOCHS:
        return "finished", EPOCHS
    return "resume", epoch + 1


state, done = training_state()
print(f"Run: {RUN_DIR}\nState: {state} ({done}/{EPOCHS} epochs complete)")

if state == "fresh":
    YOLO("yolov8n.pt").train(
        data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        seed=SEED, deterministic=True, patience=EPOCHS,
        project=str(RUNS_DIR), name=RUN_NAME, exist_ok=True, device=0, plots=True,
    )
elif state == "resume":
    YOLO(str(LAST_PT)).train(resume=True)
else:
    print(f"Training already finished - skipping. Delete {RUN_DIR} to retrain from scratch.")

assert BEST_PT.exists(), f"No best.pt at {BEST_PT} - training did not complete."

### Training curves

`results.png` plots the losses and the validation metrics per epoch. Things to look for:

* **Training losses falling** (box, cls, dfl) show the model is learning.
* **Validation mAP rising, then levelling off** is healthy. mAP that peaks and then *falls* while the
  training loss keeps falling is **overfitting**: the model is memorising the training images.
* A jump or kink in the curves around the last 10 epochs is expected. Ultralytics switches off
  **mosaic augmentation** there (`close_mosaic=10`) so the model finishes on images that look like
  the real ones.

In [ ]:
from IPython.display import Image, display

history = pd.read_csv(RUN_DIR / "results.csv")
history.columns = history.columns.str.strip()
EPOCHS_COMPLETED = len(history)
print(f"Epochs completed: {EPOCHS_COMPLETED}")
display(Image(filename=str(RUN_DIR / "results.png"), width=1000))

## Step 5 - Evaluate on VisDrone val

The fine-tuned model is scored twice, and the two answers address different questions:

1. **All 10 classes**, using Ultralytics' own `val()`. How good is this detector on VisDrone?
   This is the standard number, comparable with anyone else's YOLOv8n-on-VisDrone result.
2. **The mapped 6-class subset**, using the Step 3 evaluator with the same merge and ignore rules as
   the baseline. How much did fine-tuning help? This is the only comparison with the baseline that is
   like for like.

Some terms:

* `best.pt` is the checkpoint from the epoch with the highest validation *fitness*, a mix of mAP50
  and mAP50-95 defined by Ultralytics. It is not necessarily the last epoch.
* **Per-class precision and recall** from `val()` are read at the confidence threshold that gives
  the best F1 score, a single operating point. AP, by contrast, summarises every threshold.
* A caveat: val is also the split Ultralytics used to *choose* `best.pt`, so these numbers are
  slightly optimistic. Picking the best of 50 epochs on a set and then reporting on that same set
  flatters the result a little. It is standard practice, but worth knowing about.

In [ ]:
metrics = YOLO(str(BEST_PT)).val(
    data=str(DATA_YAML), split="val", imgsz=IMGSZ, batch=BATCH, conf=CONF, iou=NMS_IOU, max_det=MAX_DET,
    device=0, project=str(EVAL_DIR), name="finetuned_all10", exist_ok=True, plots=True,
)
box = metrics.box
FT_ALL10 = {
    "evaluator": "ultralytics val()",
    "map50": float(box.map50), "map50_95": float(box.map),
    "precision": float(box.mp), "recall": float(box.mr),
    "per_class": [
        {"class": VISDRONE_NAMES[int(c)], "gt_instances": int(gt_counts[int(c)]),
         "precision": float(box.p[k]), "recall": float(box.r[k]),
         "ap50": float(box.ap50[k]), "ap50_95": float(box.ap[k])}
        for k, c in enumerate(box.ap_class_index)
    ],
}
print(f"Fine-tuned, all 10 classes: mAP50 {FT_ALL10['map50']:.4f} | mAP50-95 {FT_ALL10['map50_95']:.4f}")
pd.DataFrame(FT_ALL10["per_class"]).round(4)

### Like for like: fine-tuned vs baseline on the mapped subset

Same images, same evaluator, same merge rule, same ignore rule. The only thing that differs is the
model.

In [ ]:
finetuned_raw = predict_all(str(BEST_PT), VAL_IMAGES)
finetuned_eval = to_eval_space(finetuned_raw, VISDRONE_TO_SUBSET)

ap, n = evaluate(finetuned_eval, GT, VISDRONE_TO_SUBSET, VISDRONE_UNSCORED, SUBSET_NAMES)
FT_SUBSET = summarize(ap, n, SUBSET_NAMES)
ap, n = evaluate(finetuned_eval, GT, VISDRONE_TO_SUBSET, (), SUBSET_NAMES)
FT_SUBSET_NO_IGNORE = summarize(ap, n, SUBSET_NAMES)

comparison = pd.DataFrame([
    {"class": b["class"], "gt_instances": b["gt_instances"],
     "baseline_ap50": b["ap50"], "finetuned_ap50": f["ap50"], "delta_ap50": f["ap50"] - b["ap50"],
     "baseline_ap50_95": b["ap50_95"], "finetuned_ap50_95": f["ap50_95"],
     "delta_ap50_95": f["ap50_95"] - b["ap50_95"]}
    for b, f in zip(BASELINE_SUBSET["per_class"], FT_SUBSET["per_class"])
])
for label, b, f in [("mAP50", BASELINE_SUBSET["map50"], FT_SUBSET["map50"]),
                    ("mAP50-95", BASELINE_SUBSET["map50_95"], FT_SUBSET["map50_95"])]:
    print(f"{label:9s} baseline {b:.4f} -> fine-tuned {f:.4f}  ({f - b:+.4f})")
print(f"(no ignore rule: baseline mAP50 {BASELINE_SUBSET_NO_IGNORE['map50']:.4f} -> "
      f"fine-tuned {FT_SUBSET_NO_IGNORE['map50']:.4f})")
comparison.round(4)

### Checking the evaluator against Ultralytics

The subset numbers come from this notebook's own evaluator, so it should not be trusted without a
check. Here it scores the fine-tuned model on **all 10 classes** with no merging and no ignoring,
which is exactly what `val()` measured above. The two will not agree to every digit, for three
reasons:

* **Matching.** This evaluator matches greedily in order of confidence, as COCO does. Ultralytics
  resolves matches by IoU.
* **Area under the curve.** The two integrate the precision-recall curve slightly differently.
* **Resizing.** `val()` resizes images in rectangular batches, while `predict()` resizes each image
  on its own.

They should be close, though: within about 0.02 mAP. A larger gap means something in the evaluator is
wrong, and the subset numbers should not be used until it is found.

In [ ]:
ap, n = evaluate(to_eval_space(finetuned_raw, IDENTITY_10), GT, IDENTITY_10, (), VISDRONE_NAMES)
ours_all10 = summarize(ap, n, VISDRONE_NAMES)
CROSSCHECK = {
    "ultralytics_map50": FT_ALL10["map50"], "notebook_map50": ours_all10["map50"],
    "ultralytics_map50_95": FT_ALL10["map50_95"], "notebook_map50_95": ours_all10["map50_95"],
}
CROSSCHECK["max_abs_diff"] = max(abs(CROSSCHECK["ultralytics_map50"] - CROSSCHECK["notebook_map50"]),
                                 abs(CROSSCHECK["ultralytics_map50_95"] - CROSSCHECK["notebook_map50_95"]))
CROSSCHECK["agrees"] = CROSSCHECK["max_abs_diff"] <= 0.02
print(json.dumps({k: round(v, 4) if isinstance(v, float) else v for k, v in CROSSCHECK.items()}, indent=2))
if not CROSSCHECK["agrees"]:
    print("WARNING: evaluators disagree by more than 0.02 mAP - treat the mapped-subset numbers as suspect.")

## Step 6 - Save the results and export the model

### The results record

The repo keeps its benchmark results in `results/*.json`. Each file is a **list of run records**, and
each record carries a UTC `timestamp`, its configuration at the top level and a `results` list, with
metrics rounded to 4 places. This record follows the same shape. It is **appended** to
`benchmarks_visdrone.json` on Drive, so a second run adds to the file instead of overwriting the first.

**To bring it into the repo**, download `MyDrive/jarvis/visdrone_yolov8n/results/benchmarks_visdrone.json`
and commit it as `results/benchmarks_visdrone.json`.

### Why export to ONNX

`best.pt` is a PyTorch checkpoint, and loading it needs PyTorch and Ultralytics. **ONNX** is a neutral
file format for the network itself, which lightweight runtimes (ONNX Runtime, TensorRT, OpenVINO) can
run without either of them. The speed step will measure fps there, so the export is done now while
the GPU session and the weights are to hand.

It is exported at a **fixed 640x640** input size (`dynamic=False`), which matches the speed target
and lets runtimes optimise for that one shape. The cell then runs the `.pt` and the `.onnx` on the
same image and checks that they find the same boxes. A broken export usually fails silently, so this
check is worth the few seconds.

Large binaries do not go in git: the repo's `.gitignore` already excludes `*.pt`. The `.onnx` file is
kept on Drive, and only its SHA-256 goes into the record.

In [ ]:
import hashlib
import shutil


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


EXPORT_PT = EXPORT_DIR / f"{RUN_NAME}.pt"
EXPORT_ONNX = EXPORT_DIR / f"{RUN_NAME}.onnx"
shutil.copy2(BEST_PT, EXPORT_PT)
onnx_out = YOLO(str(BEST_PT)).export(format="onnx", imgsz=IMGSZ, dynamic=False, simplify=True, half=False)
shutil.copy2(onnx_out, EXPORT_ONNX)

# Sanity check: .pt and .onnx should find the same boxes on the same image.
probe = VAL_IMAGES[0]
r_pt = YOLO(str(EXPORT_PT)).predict(probe, imgsz=IMGSZ, conf=0.25, device="cpu", verbose=False)[0].boxes
r_ox = YOLO(str(EXPORT_ONNX), task="detect").predict(probe, imgsz=IMGSZ, conf=0.25, device="cpu",
                                                    verbose=False)[0].boxes
iou = box_iou(r_pt.xyxy.numpy(), r_ox.xyxy.numpy())
same_cls = r_pt.cls.numpy()[:, None] == r_ox.cls.numpy()[None, :]
matched = float(((iou >= 0.9) & same_cls).any(1).mean()) if len(r_pt) else 1.0
ONNX_CHECK = {"probe_image": Path(probe).name, "boxes_pt": len(r_pt), "boxes_onnx": len(r_ox),
              "pt_boxes_matched_iou90": round(matched, 4)}
print(ONNX_CHECK)
if matched < 0.9:
    print("WARNING: ONNX output differs from the .pt more than expected - inspect before the speed step.")

ARTIFACTS = {
    "best_pt": {"path": str(EXPORT_PT.relative_to(DRIVE_ROOT)), "sha256": sha256(EXPORT_PT)},
    "onnx": {"path": str(EXPORT_ONNX.relative_to(DRIVE_ROOT)), "sha256": sha256(EXPORT_ONNX),
             "input_shape": [1, 3, IMGSZ, IMGSZ], "dynamic": False, "half": False},
}
print(json.dumps(ARTIFACTS, indent=2))

### Write the record

In [ ]:
import platform

import onnx
import onnxruntime


def r4(x):
    # Round like the repo's results files; NaN (a class with no val instances) becomes null.
    if isinstance(x, dict):
        return {k: r4(v) for k, v in x.items()}
    if isinstance(x, list):
        return [r4(v) for v in x]
    if isinstance(x, float):
        return None if np.isnan(x) else round(x, 4)
    return x


sessions = [json.loads(line) for line in SESSIONS_LOG.read_text().splitlines() if line.strip()]
record = r4({
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "dataset": "visdrone2019-det",
    "split": "val",
    "num_images": len(GT),
    "skipped_images": SKIPPED,
    "seed": SEED,
    "versions": {
        "python": platform.python_version(), "ultralytics": ultralytics.__version__,
        "torch": torch.__version__, "torchvision": torchvision.__version__,
        "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
        "numpy": np.__version__, "onnx": onnx.__version__, "onnxruntime": onnxruntime.__version__,
    },
    "gpu": GPU_NAME,
    "gpu_sessions": sessions,
    "model": "yolov8n",
    "init_weights": "yolov8n.pt (COCO)",
    "epochs": EPOCHS,
    "epochs_completed": EPOCHS_COMPLETED,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "eval": {"conf": CONF, "nms_iou": NMS_IOU, "max_det": MAX_DET,
             "iou_thresholds": "0.50:0.05:0.95", "ap_interpolation": "101-point"},
    "mapped_subset": {
        "classes": SUBSET_NAMES,
        "coco_to_subset": {"person": "person", "bicycle": "bicycle", "car": "car",
                           "truck": "truck", "bus": "bus", "motorcycle": "motor"},
        "visdrone_to_subset": {VISDRONE_NAMES[k]: SUBSET_NAMES[v] for k, v in VISDRONE_TO_SUBSET.items()},
        "unscored": {name: "no COCO equivalent - not scored for either model on this subset"
                     for name in VISDRONE_UNSCORED.values()},
        "unscored_policy": "ignore regions: an unmatched prediction overlapping an unscored-class box at the "
                           "IoU threshold is neither TP nor FP (applied to both models)",
        "evaluator": "notebook, greedy COCO matching, re-NMS after class merge",
    },
    "results": [
        {"model": "baseline", "weights": "yolov8n.pt", "trained_on": "coco",
         "mapped_subset": BASELINE_SUBSET,
         "mapped_subset_no_ignore": {k: BASELINE_SUBSET_NO_IGNORE[k] for k in ("map50", "map50_95")},
         "all_classes": None,
         "all_classes_note": "not scored: van, tricycle, awning-tricycle have no COCO class"},
        {"model": "finetuned", "weights": ARTIFACTS["best_pt"]["path"], "trained_on": "visdrone2019-det train",
         "mapped_subset": FT_SUBSET,
         "mapped_subset_no_ignore": {k: FT_SUBSET_NO_IGNORE[k] for k in ("map50", "map50_95")},
         "all_classes": FT_ALL10},
    ],
    "evaluator_crosscheck": CROSSCHECK,
    "onnx_check": ONNX_CHECK,
    "artifacts": ARTIFACTS,
})

history_records = json.loads(RESULTS_JSON.read_text()) if RESULTS_JSON.exists() else []
history_records.append(record)
RESULTS_JSON.write_text(json.dumps(history_records, indent=2) + "\n")
print(f"Appended run {len(history_records)} to {RESULTS_JSON}")
print(json.dumps({k: record[k] for k in ("timestamp", "seed", "epochs_completed", "imgsz", "gpu")}, indent=2))

## Wrap-up

What exists on Drive now (`MyDrive/jarvis/visdrone_yolov8n/`):

* `runs/<run>/`: the full training run, including `results.csv`, the curves and the confusion matrix
* `eval/finetuned_all10/`: the validation plots (PR curves, confusion matrix)
* `export/<run>.pt` and `export/<run>.onnx`: the inputs to the speed step
* `results/benchmarks_visdrone.json`: the record to copy into the repo's `results/`

What this notebook **did not** measure: **speed**. No fps or latency number comes from here. Colab
GPUs vary from session to session and are not the target hardware, so timings taken here would not
mean much. That is the next step, and it runs against the ONNX file exported above.